In [1]:
import torch
import torch.nn as nn

In [2]:
class SimpleModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.projector = nn.Linear(3, 16)
        self.ffn = nn.Sequential(
            nn.Linear(16, 32),
            nn.ReLU(),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 1)
        )
        self.loss_fn = nn.BCEWithLogitsLoss()

    def forward(self, runs, wickets, balls, labels):
        x = torch.stack([runs, wickets, balls], dim=1).float()
        x = self.projector(x)
        x = self.ffn(x)

        logits = x.squeeze(1)
        loss = self.loss_fn(logits, labels.float())
        return {"loss": loss, "logits": logits}


In [3]:
from pathlib import Path
import sys

project_root = Path.cwd()
if (project_root / "src").is_dir():
    pass
elif (project_root / "cricket-win-predict" / "src").is_dir():
    project_root = project_root / "cricket-win-predict"
elif project_root.name == "notebooks" and (project_root.parent / "src").is_dir():
    project_root = project_root.parent
else:
    raise FileNotFoundError("Could not locate the cricket-win-predict project root")

sys.path.insert(0, str(project_root))

In [4]:
from src.data.filter import load_data, filter_with_nation_winners
from src.data.preprocess import process_data

data = load_data()
filtered_data = filter_with_nation_winners(data)
processed_data = process_data(filtered_data)[1]

In [5]:
from sklearn.model_selection import train_test_split

train_data, test_data = train_test_split(processed_data, test_size=0.2, random_state=42)

In [7]:
from transformers import Trainer, TrainingArguments

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = torch.sigmoid(torch.tensor(logits)).numpy() > 0.5
    accuracy = (predictions == labels).mean()
    return {"accuracy": accuracy}

train_args = TrainingArguments(
    per_device_train_batch_size=32,
    per_device_eval_batch_size=5000,
    num_train_epochs=2,
    learning_rate=1e-4,
    output_dir=str(project_root / "models" / "simple_model"),
    logging_strategy="steps",
    logging_steps=1000,
    eval_strategy="steps",
    eval_steps=1000,
)

model = SimpleModel()

trainer = Trainer(
    model=model,
    args=train_args,
    train_dataset=train_data,
    eval_dataset=test_data,
)

trainer.train()

Step,Training Loss,Validation Loss
1000,0.510589,0.436306
2000,0.424714,0.421775
3000,0.410956,0.406129
4000,0.401206,0.397876
5000,0.391788,0.395158
6000,0.393458,0.394125
7000,0.392576,0.392498
8000,0.393215,0.391995
9000,0.390065,0.388766
10000,0.387169,0.386225


TrainOutput(global_step=26240, training_loss=0.39115757651445343, metrics={'train_runtime': 186.2333, 'train_samples_per_second': 4508.667, 'train_steps_per_second': 140.899, 'total_flos': 0.0, 'train_loss': 0.39115757651445343, 'epoch': 2.0})